In [8]:
import polars as pl
import json

# Create table of places

## Load data for places

In [9]:
places = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/places.csv"
)
places_to_types = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/places_to_types.csv"
)
feature_types = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/feature_types.csv"
)
event_types = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/events_to_types.csv"
)
locs = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/locations.csv"
)
places_to_attestations = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/places_to_attestation.csv"
)
places_to_establishment = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/places_to_establishment.csv",
    ignore_errors=True,
)
feature_class = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/feature_class.csv"
)
# Remove duplicate place entries
locs = locs.unique(subset=["place_id"], keep="first")

## Join tables to create places table

In [10]:
from re import sub


pt_df = places.join(places_to_types, on="place_id", how="left")
pt_df = pt_df.join(locs, on="place_id", how="left").unique(
    subset=["place_id"], keep="first"
)
pt_df = pt_df.rename({"ft_id": "yft_id"})
pt_df = pt_df.join(feature_types, on="yft_id", how="left", suffix="_ft_tp")
pt_df = pt_df.join(places_to_attestations, on="place_id", how="left")
pt_df = pt_df.join(places_to_establishment, on="place_id", how="left")
pt_df = pt_df.join(feature_class, on="fct_id", how="left")
pt_df = pt_df.select(
    [
        "place_id",
        "tr_title",
        "ch_pinyin",
        "latitude",
        "longitude",
        "attestation",
        "ch_title",
        "feature_class",
        "en_title",
        "est_year",
    ]
)
pt_df = pt_df.unique()
pt_df.head()

place_id,tr_title,ch_pinyin,latitude,longitude,attestation,ch_title,feature_class,en_title,est_year
str,str,str,f64,f64,str,str,str,str,i64
"""yrdb1420""","""李鋼堡""","""ligang bao""",38.620131,106.398767,"""us_80453""","""堡""","""habitations""","""fortress""",1582
"""yrdb786""","""寧安堡""","""ningan bao""",37.366209,105.644327,"""us_70201""","""堡""","""habitations""","""fortress""",1820
"""yrdb297""","""華家嶺""","""huajia ling""",35.374803,104.997185,"""us_80512""","""嶺""","""natural features""","""ridge""",1820
"""yrdb3121""","""陽谷堤""","""yanggu di""",null,null,"""ds_748""","""堤""","""water control management""","""dike""",null
"""yrdb978""","""巴爾蘇海""","""baersuhai""",39.508078,109.299693,"""us_81020""",null,null,null,1820


In [11]:
print(places.shape)
print(pt_df.shape)
pt_df = pt_df.sort("est_year", nulls_last=True).unique(
    subset=["place_id"], keep="first"
)
print(pt_df.shape)

(4480, 3)
(5074, 10)
(4480, 10)


## Split into Upstream and Downstream data

In [12]:
# upstream = pt_df.filter(pl.col("attestation").str.starts_with("us"))#
# downstream = pt_df.filter(pl.col("attestation").str.starts_with("ds"))

In [13]:
upstream = pt_df.filter(
    pl.col("attestation")
      .is_not_null()
      .and_(
          pl.col("attestation")
            .str.to_lowercase()
            .str.contains("us_|fort")
      )
)

downstream = pt_df.filter(
    pl.col("attestation")
      .is_not_null()
      .and_(
          pl.col("attestation")
            .str.to_lowercase()
            .str.contains("ds_")
      )
)

In [14]:
places_to_attestations["attestation_id"].count()

5992

In [15]:
total_places_w_att = places_to_attestations["attestation_id"].count()

In [16]:
num_ds = places_to_attestations["attestation_id"].str.contains("us").sum()

In [17]:
num_us = places_to_attestations["attestation_id"].str.contains("ds").sum()

In [18]:
num_ds + num_us == total_places_w_att

False

In [19]:
total_places_w_att - (num_ds + num_us)

1136

In [20]:
downstream_map = {y: 0 for y in downstream["est_year"].unique()}
for y in downstream["est_year"]:
    if y in downstream_map:
        downstream_map[y] += 1
    else:
        downstream_map[y] = 1
downstream_map

{None: 1527,
 -256: 4,
 327: 1,
 497: 1,
 612: 1,
 741: 1,
 1111: 16,
 1582: 4,
 1820: 1}

In [21]:
total = {y: 0 for y in upstream["est_year"].unique()}
for y in pt_df["est_year"]:
    if y in total:
        total[y] += 1
    else:
        total[y] = 1
total

{None: 1530,
 -256: 142,
 -206: 19,
 9: 73,
 220: 89,
 262: 34,
 281: 50,
 327: 10,
 366: 4,
 382: 3,
 395: 4,
 497: 121,
 546: 6,
 572: 14,
 612: 58,
 741: 183,
 1111: 643,
 1189: 238,
 1330: 85,
 1449: 1,
 1582: 325,
 1820: 848}

In [22]:
upstream_map = {y: 0 for y in upstream["est_year"].unique()}
for y in upstream["est_year"]:
    if y in upstream_map:
        upstream_map[y] += 1
    else:
        upstream_map[y] = 1
upstream_map

{None: 3,
 -256: 138,
 -206: 19,
 9: 73,
 220: 89,
 262: 34,
 281: 50,
 327: 9,
 366: 4,
 382: 3,
 395: 4,
 497: 120,
 546: 6,
 572: 14,
 612: 57,
 741: 182,
 1111: 627,
 1189: 238,
 1330: 85,
 1449: 1,
 1582: 321,
 1820: 847}

In [23]:
import requests

# Join Upstream places with infromation
eras = requests.get(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/eras.geojson"
).json()["features"]
dyns = requests.get(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/dynasties.geojson"
).json()["features"]
dyns_df = pl.DataFrame([f["properties"] for f in dyns], orient="row")
eras_df = pl.DataFrame([f["properties"] for f in eras], orient="row")
dyns_df.head()

id,name_en,name_ch,start_cert,start_date,end_cert,end_date
i64,str,str,str,i64,str,i64
1,"""Xia dynasty ""","""夏朝""","""n""",-2070,"""n""",-1600
2,"""Shang dynasty ""","""商朝 ""","""n""",-1600,"""n""",-1046
3,"""Zhou dynasty ""","""周朝 ""","""n""",-1046,"""y""",-256
4,"""Qin dynasty ""","""秦朝 ""","""y""",-221,"""y""",-207
5,"""Han dynasty ""","""漢朝 ""","""y""",-202,"""y""",220


In [24]:
eras_df.head()

id,monarch,era_name_en,era_name_ch,era_start,era_end
str,str,str,str,i64,i64
"""I0019""","""I0001""","""Jianlong""","""建隆""",960,963
"""I0020""","""I0001""","""Qiande""","""乾德""",963,968
"""I0021""","""I0001""","""Kaibao""","""開寶""",968,976
"""I0022""","""I0002""","""Taipingxingguo""","""太平興國""",976,984
"""I0023""","""I0002""","""Yongxi""","""雍熙""",984,987


In [25]:
dyns_df = dyns_df.select(["name_en", "name_ch", "start_date", "end_date"]).rename(
    {"name_en": "dynasty_en", "name_ch": "dynasty_ch"}
)

In [26]:
dyn_en = []
dyn_ch = []
for row in upstream.iter_rows():
    est_year = row[9]
    if est_year is None:
        dyn_en.append(None)
        dyn_ch.append(None)
        continue
    matched_dyn = dyns_df.filter(
        (pl.col("start_date") <= est_year) & (pl.col("end_date") >= est_year)
    )
    if matched_dyn.is_empty():
        dyn_en.append(None)
        dyn_ch.append(None)
    else:
        dyn_en.append(matched_dyn[0, "dynasty_en"].strip())
        dyn_ch.append(matched_dyn[0, "dynasty_ch"].strip())

upstream = upstream.with_columns(
    [pl.Series("dynasty_en", dyn_en), pl.Series("dynasty_ch", dyn_ch)]
)

upstream.head()

place_id,tr_title,ch_pinyin,latitude,longitude,attestation,ch_title,feature_class,en_title,est_year,dynasty_en,dynasty_ch
str,str,str,f64,f64,str,str,str,str,i64,str,str
"""yrdb4423""","""承平寨""","""chengping zhai""",37.356006,109.851922,"""Fortifications_Year1189_239""",null,null,null,1189,"""Western Liao""","""西遼"""
"""yrdb4753""","""西壕寨""","""Xihao""",35.808376,107.186897,"""Fortifications_Year1189""",null,null,null,1189,"""Western Liao""","""西遼"""
"""yrdb4534""","""通西寨""","""tongxi zhai""",35.383316,104.63228,"""Fortifications_Year741_31""",null,null,null,741,"""Tang dynasty""","""唐朝"""
"""yrdb4831""","""赤糖 關""","""Chitang""",38.176768,112.624381,"""Fortifications_Year741""",null,null,null,741,"""Tang dynasty""","""唐朝"""
"""yrdb3178""","""陘城""","""xing cheng""",35.678231,111.457893,"""us_10050""","""城""","""habitations""","""walled settlement""",-256,"""Zhou dynasty""","""周朝"""


In [27]:
# convert upstream to json and write to upstream-data.json

upstream_data = []

for row in upstream.iter_rows():
    id = row[0]
    hz = row[1]
    py = row[2]
    x_coor = row[4]
    y_coor = row[3]
    date = row[9]
    regime = row[10]
    regime_ch = row[11]
    name_type = row[6]
    name_type_en = row[8]
    name_class_en = row[7]

    place_json = {
        "id": id,
        "hz": hz,
        "py": py,
        "x_coor": x_coor,
        "y_coor": y_coor,
        "date": date,
        "regime": regime,
        "regime_ch": regime_ch,
        "name_type": name_type,
        "name_type_en": name_type_en,
        "name_class_en": name_class_en,
    }
    upstream_data.append(place_json)

with open("upstream-data.json", "w", encoding="utf-8") as f:
     json.dump(upstream_data, f, ensure_ascii=False, indent=4)

# Create table for events

## Load data for events

In [28]:
import numpy as np

events = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/events.csv"
)
event_types = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/event_types.csv"
)
event_cats = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/event_categories.csv"
)

events_to_places = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/events_to_places.csv"
).rename({"event": "event_id"})
events_to_types = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/events_to_types.csv"
)
events_to_sources = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/sources_to_events.csv",
    ignore_errors=True,
)
sources = pl.read_csv(
    "https://raw.githubusercontent.com/YellowRiverDatabase/geodata/refs/heads/main/relational-data/sources.csv",
    null_values=["", "nan", "NaN", "NA", "null"],
    ignore_errors=True,
)
events_to_sources.head()

src_to_evt_id,source_id,event_id
str,i64,str
"""srev_1""",11001001,"""ev_1"""
"""srev_2""",11001002,"""ev_2"""
"""srev_3""",11001003,"""ev_3"""
"""srev_4""",11001004,"""ev_4"""
"""srev_5""",11001005,"""ev_5"""


In [29]:
events.head(1)

event_id,ch_date,western_date,description,notes
str,str,f64,str,str
"""ev_1""","""史前时代""",-2356.0,null,null


In [30]:
event_types.head(1)

event_type_id,zh_ch_title,en_title,en_type,evc_id,description
str,str,str,str,str,str
"""evtype_1""","""溢""","""yi""","""Flood""","""evc_1""","""Any time there is a zhang 漲(ra…"


In [31]:
events_to_places.head(1)

evtp_id,place_id,event_id,attestation
str,str,str,str
"""evtp_1""","""yrdb2""","""ev_858""","""ds_1019"""


In [32]:
events_to_types.head(1)

evetotyp_id,event_id,event_type_id
str,str,str
"""evetotyp_1""","""ev_1""","""evtype_1"""


In [33]:
event_cats.head(1)

evc_id,zh_cn_category,en_category
str,str,str
"""evc_1""","""水災""","""Disasters"""


In [34]:
sources.head(1)

source_id,source,page,chinese_date,western_date,old_placename_chinese,modern_placename_chinese,event_type_chinese,event_name,event_description,primary_source_1,primary_source_2,notes
i64,str,str,str,str,str,str,str,str,str,str,str,str
10100001,"""HDSJ""",null,"""传说时代""","""约-21世纪初""",null,null,null,"""大禹治水""","""传说中的尧舜时代，黄河流域发生大洪水，为制止洪水泛滥，尧召集…",null,null,null


In [35]:
print(f"events length: {events.shape}")
ev_df = events.join(events_to_types, on="event_id", how="left", suffix="_ett")
ev_df = ev_df.join(event_types, on="event_type_id", how="left")
print(f"ev_df length: {ev_df.shape}")
# wanted columsn from event_types: zh_ch_title,	en_title, en_type
# wanted columns from events: event_id	ch_date	western_date	description	notes
ev_df = ev_df.select(
    [
        "event_id",
        "ch_date",
        "western_date",
        "description",
        "notes",
        "zh_ch_title",
        "en_title",
        "evc_id",
        "en_type",
    ]
)
ev_df = ev_df.rename(
    {
        "ch_date": "event_date_ch",
        "western_date": "event_date_western",
        "description": "event_description",
        "notes": "event_notes",
        "zh_ch_title": "event_type_ch",
        "en_title": "event_type_py",
        "en_type": "event_type_en",
    }
)
print(f"ev_df length: {ev_df.shape}")
ev_df.head()

events length: (3754, 5)
ev_df length: (5349, 12)
ev_df length: (5349, 9)


event_id,event_date_ch,event_date_western,event_description,event_notes,event_type_ch,event_type_py,evc_id,event_type_en
str,str,f64,str,str,str,str,str,str
"""ev_1""","""史前时代""",-2356.0,null,null,"""溢""","""yi""","""evc_1""","""Flood"""
"""ev_2""","""史前时代""",-2356.0,null,null,"""溢""","""yi""","""evc_1""","""Flood"""
"""ev_3""","""史前时代""",-2356.0,null,null,"""溢""","""yi""","""evc_1""","""Flood"""
"""ev_3""","""史前时代""",-2356.0,null,null,"""災""","""zai""","""evc_1""","""Disaster"""
"""ev_4""","""史前时代""",-2356.0,null,null,"""溢""","""yi""","""evc_1""","""Flood"""


In [36]:
ev_df = ev_df.join(event_cats, on="evc_id", how="left")
ev_df.head()

event_id,event_date_ch,event_date_western,event_description,event_notes,event_type_ch,event_type_py,evc_id,event_type_en,zh_cn_category,en_category
str,str,f64,str,str,str,str,str,str,str,str
"""ev_1""","""史前时代""",-2356.0,null,null,"""溢""","""yi""","""evc_1""","""Flood""","""水災""","""Disasters"""
"""ev_2""","""史前时代""",-2356.0,null,null,"""溢""","""yi""","""evc_1""","""Flood""","""水災""","""Disasters"""
"""ev_3""","""史前时代""",-2356.0,null,null,"""溢""","""yi""","""evc_1""","""Flood""","""水災""","""Disasters"""
"""ev_3""","""史前时代""",-2356.0,null,null,"""災""","""zai""","""evc_1""","""Disaster""","""水災""","""Disasters"""
"""ev_4""","""史前时代""",-2356.0,null,null,"""溢""","""yi""","""evc_1""","""Flood""","""水災""","""Disasters"""


In [37]:
ev_df = ev_df.rename(
    {"zh_cn_category": "type_category_ch", "en_category": "type_category_en"}
)

In [38]:
# group by event_id and aggregate types columns into lists
ev_df = ev_df.group_by(["event_id", "event_date_western"]).agg(
    pl.col("event_date_ch").first(),
    pl.col("event_description").first(),
    pl.col("event_notes").first(),
    pl.col("event_type_ch").implode(),
    pl.col("event_type_py").implode(),
    pl.col("event_type_en").implode(),
    pl.col("evc_id").implode(),
    pl.col("type_category_ch").implode(),
    pl.col("type_category_en").implode(),
)
print(f"original events length: {events.shape}")
print(f"ev_df length after groupby: {ev_df.shape}")
ev_df.head()

original events length: (3754, 5)
ev_df length after groupby: (3754, 11)


event_id,event_date_western,event_date_ch,event_description,event_notes,event_type_ch,event_type_py,event_type_en,evc_id,type_category_ch,type_category_en
str,f64,str,str,str,list[str],list[str],list[str],list[str],list[str],list[str]
"""ev_63""",-500.0,"""战国时期""","""record of Yellow River change …",null,"[""議""]","[""yi""]","[""Proposals and Discussion""]","[""evc_2""]","[""水利""]","[""Management""]"
"""ev_2392""",1677.0,"""清圣祖康熙十六年""",null,null,"[""建"", ""修""]","[""jian"", ""xiu""]","[""New Construction"", ""Repair of Structures""]","[""evc_2"", ""evc_2""]","[""水利"", ""水利""]","[""Management"", ""Management""]"
"""ev_3580""",1887.0,"""光绪十三年""",null,null,"[""決"", ""溢""]","[""jue"", ""yi""]","[""Breach"", ""Flood""]","[""evc_1"", ""evc_1""]","[""水災"", ""水災""]","[""Disasters"", ""Disasters""]"
"""ev_467""",841.0,"""唐武宗会昌元年""","""贵德流沙""",null,"[""災""]","[""zai""]","[""Disaster""]","[""evc_1""]","[""水災""]","[""Disasters""]"
"""ev_2385""",1677.0,"""康熙十六年""","""Jin Fu appointed as the new su…",null,"[""治""]","[""zhi""]","[""Management""]","[""evc_2""]","[""水利""]","[""Management""]"


In [39]:
# Join Sources
print(f"original events length: {events.shape}")
ev_df = ev_df.join(events_to_sources, on="event_id", how="left")
ev_df = ev_df.join(sources, on="source_id", how="left")
# ev_df columns: event_id, event_date_western, event_date_ch	event_description, event_notes, event_type_ch, event_type_py, event_type_en
# source columns: source, page, chinese_date, western_date, old_placename_chinese, modern_placename_chinese, event_type_chinese, event_name, event_description, primary_source_1, primary_source_2, notes
ev_df = ev_df.rename(
    {
        "page": "source_page",
        "chinese_date": "source_ch_date",
        "western_date": "source_western_date",
        "event_type_chinese": "source_event_type_chinese",
        "event_name": "source_event_name",
        "event_description_right": "source_event_description",
    }
)
ev_df.head()

original events length: (3754, 5)


event_id,event_date_western,event_date_ch,event_description,event_notes,event_type_ch,event_type_py,event_type_en,evc_id,type_category_ch,type_category_en,src_to_evt_id,source_id,source,source_page,source_ch_date,source_western_date,old_placename_chinese,modern_placename_chinese,source_event_type_chinese,source_event_name,source_event_description,primary_source_1,primary_source_2,notes
str,f64,str,str,str,list[str],list[str],list[str],list[str],list[str],list[str],str,i64,str,str,str,str,str,str,str,str,str,str,str,str
"""ev_63""",-500.0,"""战国时期""","""record of Yellow River change …",null,"[""議""]","[""yi""]","[""Proposals and Discussion""]","[""evc_2""]","[""水利""]","[""Management""]","""srev_66""",10900022,"""SWDS""",null,"""战国时期""","""-5世纪～-3世纪""",null,null,null,"""(1-20) 《逸周书》记黄河水文变化""","""成书于战国时代的《逸周书·时训解》中记有黄河流域的水文季节变…",null,null,null
"""ev_2392""",1677.0,"""清圣祖康熙十六年""",null,null,"[""建"", ""修""]","[""jian"", ""xiu""]","[""New Construction"", ""Repair of Structures""]","[""evc_2"", ""evc_2""]","[""水利"", ""水利""]","[""Management"", ""Management""]","""srev_3685""",10410145,"""HHNB""",null,"""清圣祖康熙十六年""","""1677""",null,null,"""修""",null,"""靳辅大筑黄河缕堤；南岸自白洋河至云梯关二百三十里，北岸自清河…","""云梯关以上有原未有堤者，有原有堤而今全无土者，有原有堤而今缺…",null,null
"""ev_3580""",1887.0,"""光绪十三年""",null,null,"[""決"", ""溢""]","[""jue"", ""yi""]","[""Breach"", ""Flood""]","[""evc_1"", ""evc_1""]","[""水災"", ""水災""]","[""Disasters"", ""Disasters""]","""srev_5016""",10300615,"""LYZS""","""425""","""光绪十三年""","""1887""",null,null,null,"""黄河流域大水决溢""","""正月长清县谯家庄，济阳县郭家纸坊、王家圈、韩家寺，齐河县纸坊…","""《再续》, 《潘镒芬稿》, 《调查材料》, 《直隶河防辑要》…",null,null
"""ev_467""",841.0,"""唐武宗会昌元年""","""贵德流沙""",null,"[""災""]","[""zai""]","[""Disaster""]","[""evc_1""]","[""水災""]","[""Disasters""]","""srev_718""",10100075,"""HDSJ""",null,"""唐武宗会昌元年""","""841""",null,null,null,"""贵德流沙""","""廓州(今贵德)流沙(注：泥石流)渡黄河南，冲压忙拉川城堡，房…",null,null,null
"""ev_2385""",1677.0,"""康熙十六年""","""Jin Fu appointed as the new su…",null,"[""治""]","[""zhi""]","[""Management""]","[""evc_2""]","[""水利""]","[""Management""]","""srev_3676""",10100214,"""HDSJ""",null,"""康熙十六年""","""1677""",null,null,null,"""靳辅出任河督""","""二月，河督王光裕被撤职查问，以安徽巡抚靳辅接任河道总督。三月…",null,null,null


In [40]:
ev_df = ev_df.select(
    [
        "event_id",
        "event_date_western",
        "event_date_ch",
        "event_description",
        "event_notes",
        "event_type_ch",
        "event_type_py",
        "event_type_en",
        "evc_id",
        "source",
        "source_page",
        "source_ch_date",
        "source_western_date",
        "source_event_type_chinese",
        "source_event_name",
        "source_event_description",
        "type_category_ch",
        "type_category_en",
    ]
)
print(f"ev_df length after join2: {ev_df.shape}")
ev_df.head()

ev_df length after join2: (5235, 18)


event_id,event_date_western,event_date_ch,event_description,event_notes,event_type_ch,event_type_py,event_type_en,evc_id,source,source_page,source_ch_date,source_western_date,source_event_type_chinese,source_event_name,source_event_description,type_category_ch,type_category_en
str,f64,str,str,str,list[str],list[str],list[str],list[str],str,str,str,str,str,str,str,list[str],list[str]
"""ev_63""",-500.0,"""战国时期""","""record of Yellow River change …",null,"[""議""]","[""yi""]","[""Proposals and Discussion""]","[""evc_2""]","""SWDS""",null,"""战国时期""","""-5世纪～-3世纪""",null,"""(1-20) 《逸周书》记黄河水文变化""","""成书于战国时代的《逸周书·时训解》中记有黄河流域的水文季节变…","[""水利""]","[""Management""]"
"""ev_2392""",1677.0,"""清圣祖康熙十六年""",null,null,"[""建"", ""修""]","[""jian"", ""xiu""]","[""New Construction"", ""Repair of Structures""]","[""evc_2"", ""evc_2""]","""HHNB""",null,"""清圣祖康熙十六年""","""1677""","""修""",null,"""靳辅大筑黄河缕堤；南岸自白洋河至云梯关二百三十里，北岸自清河…","[""水利"", ""水利""]","[""Management"", ""Management""]"
"""ev_3580""",1887.0,"""光绪十三年""",null,null,"[""決"", ""溢""]","[""jue"", ""yi""]","[""Breach"", ""Flood""]","[""evc_1"", ""evc_1""]","""LYZS""","""425""","""光绪十三年""","""1887""",null,"""黄河流域大水决溢""","""正月长清县谯家庄，济阳县郭家纸坊、王家圈、韩家寺，齐河县纸坊…","[""水災"", ""水災""]","[""Disasters"", ""Disasters""]"
"""ev_467""",841.0,"""唐武宗会昌元年""","""贵德流沙""",null,"[""災""]","[""zai""]","[""Disaster""]","[""evc_1""]","""HDSJ""",null,"""唐武宗会昌元年""","""841""",null,"""贵德流沙""","""廓州(今贵德)流沙(注：泥石流)渡黄河南，冲压忙拉川城堡，房…","[""水災""]","[""Disasters""]"
"""ev_2385""",1677.0,"""康熙十六年""","""Jin Fu appointed as the new su…",null,"[""治""]","[""zhi""]","[""Management""]","[""evc_2""]","""HDSJ""",null,"""康熙十六年""","""1677""",null,"""靳辅出任河督""","""二月，河督王光裕被撤职查问，以安徽巡抚靳辅接任河道总督。三月…","[""水利""]","[""Management""]"


In [41]:
# Group and aggregate sources into lists
ev_df = ev_df.group_by(
    [
        "event_id",
        "event_date_western",
        "event_date_ch",
        "event_description",
        "event_notes",
        "evc_id",
        "event_type_ch",
        "event_type_py",
        "event_type_en",
        "type_category_ch",
        "type_category_en",
    ]
).agg(
    [
        pl.col("source").implode(),
        pl.col("source_page").implode(),
        pl.col("source_ch_date").implode(),
        pl.col("source_western_date").implode(),
        pl.col("source_event_type_chinese").implode(),
        pl.col("source_event_name").implode(),
        pl.col("source_event_description").implode(),
    ]
)

In [42]:
# Create formatted citation from paired lists
ev_df1 = ev_df.with_columns(
    [
        pl.struct(
            [
                "source",
                "source_western_date",
                "source_ch_date",
                "source_event_description",
                "source_page",
            ]
        )
        .map_elements(
            lambda x: "; ".join(
                [
                    f"""{name}{', in ' + west if west is not None else ''} {'('+ch+')' if ch is not None else ''} {'describes it as ' + description if description is not None else ''} {'(p. ' + page + ')' if page is not None else ''}""".strip()
                    for name, west, ch, description, page in zip(
                        x["source"],
                        x["source_western_date"],
                        x["source_ch_date"],
                        x["source_event_description"],
                        x["source_page"],
                    )
                    if name is not None
                ]
            ),
            return_dtype=pl.String,
        )
        .alias("citation")
    ]
)
print(f"ev_df length after citations: {ev_df1.shape}")
ev_df1.head()

ev_df length after citations: (3754, 19)


event_id,event_date_western,event_date_ch,event_description,event_notes,evc_id,event_type_ch,event_type_py,event_type_en,type_category_ch,type_category_en,source,source_page,source_ch_date,source_western_date,source_event_type_chinese,source_event_name,source_event_description,citation
str,f64,str,str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str
"""ev_2015""",1588.0,"""明神宗万历十六年""",null,null,"[""evc_2""]","[""議""]","[""yi""]","[""Proposals and Discussion""]","[""水利""]","[""Management""]","[""HHNB""]",[null],"[""明神宗万历十六年""]","[""1588""]","[""条陈""]",[null],"[""六月己未勘科常居敬奏：“黄河故道开复甚难，宜罢役。”而訾家营支河之议起。（明神宗实录）""]","""HHNB, in 1588 (明神宗万历十六年) descr…"
"""ev_3548""",1884.0,"""光绪十年""",null,null,"[""evc_1"", ""evc_2"", ""evc_2""]","[""決"", ""救"", ""修""]","[""jue"", ""jiu"", ""xiu""]","[""Breach"", ""Emergency Repair"", ""Repair of Structures""]","[""水災"", ""水利"", ""水利""]","[""Disasters"", ""Management"", ""Management""]","[""SLSY""]","[""388""]","[""光绪十年""]","[""1884""]",[null],[null],"[""吴元炳奏：东省今岁缕堤、遥堤所决各口，如东阿之三里庄、吴家坝、史家桥、陶城埠、张秋镇、挂剑台、郎家营、于家庄，齐河之红庙、李家岸、柳家屯、姚吕庄，历城之蒋家庄、北小街、霍家溜、河套蠲、纸坊、冯家庄，章丘之罗家庄，齐东之萧家庄、东月堤、西月堤、许家圈、盛家庄、大张家庄、邵家庄、生家庄，利津之张家滩、卞家庄、张家庄、宁海等处。《再续行水金鉴》引《山东河工成案》闰五月又决利津县北十四户。《治水述要》七月，南岸中汛东明高村失事，旋即堵合。《直隶河防辑要》""]","""SLSY, in 1884 (光绪十年) describes…"
"""ev_3746""",1909.0,"""清宣统元年""",null,null,"[""evc_1"", ""evc_2""]","[""決"", ""修""]","[""jue"", ""xiu""]","[""Breach"", ""Repair of Structures""]","[""水災"", ""水利""]","[""Disasters"", ""Management""]","[""ZDZH"", ""SLSY""]","[""299"", ""395""]","[""清宣统元年"", ""宣统元年""]","[""1909"", ""1909""]","[""黄河决溢"", null]","[null, null]","[""决开州孟民庄。明年塞。"", ""决开州孟民庄。《清史·河渠志》濮州北岸马刘家开口。《黄河志》""]","""ZDZH, in 1909 (清宣统元年) describe…"
"""ev_1375""",1301.0,"""大德五年""",null,null,"[""evc_1""]","[""溢""]","[""yi""]","[""Flood""]","[""水災""]","[""Disasters""]","[""SLYJ"", ""LYZS""]","[""238"", ""399""]","[""大德五年"", ""大德五年""]","[""1301"", ""1301""]","[null, null]","[null, ""黄河流域大水决溢""]","[""归德水，汴梁旱，淝、汝、河溢"", ""六月济宁、东平、济南等郡水。归德水，汴梁旱，淝、汝河溢。五月济南路大水，归德府徐州、邳州睢、宁县雨五十日，沂、武二河合流，水大溢""]","""SLYJ, in 1301 (大德五年) describes…"
"""ev_1547""",1366.0,"""至正二十六年""",null,null,"[""evc_1""]","[""溢""]","[""yi""]","[""Flood""]","[""水災""]","[""Disasters""]","[""SLYJ""]","[""256""]","[""至正二十六年""]","[""1366""]",[null],[null],"[""七月，介休县汾水溢，卫辉、汴梁、钧州大水，灾""]","""SLYJ, in 1366 (至正二十六年) describ…"


In [43]:
print(f"ev_df length: {ev_df.shape}")
ev_df2 = ev_df1.join(events_to_places, on="event_id", how="left", suffix="_etp")

print(f"ev_df2 len after join places: {ev_df2.shape}")

ev_df length: (3754, 18)
ev_df2 len after join places: (10350, 22)


In [44]:
ev_df2.filter(pl.col("place_id").is_not_null()).count()

event_id,event_date_western,event_date_ch,event_description,event_notes,evc_id,event_type_ch,event_type_py,event_type_en,type_category_ch,type_category_en,source,source_page,source_ch_date,source_western_date,source_event_type_chinese,source_event_name,source_event_description,citation,evtp_id,place_id,attestation
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
9955,9955,9955,995,379,9955,9955,9955,9955,9955,9955,9955,9955,9955,9955,9955,9955,9955,9955,9955,9955,9955


In [45]:
ev_df2.filter(pl.col("attestation").str.starts_with("us")).count()

event_id,event_date_western,event_date_ch,event_description,event_notes,evc_id,event_type_ch,event_type_py,event_type_en,type_category_ch,type_category_en,source,source_page,source_ch_date,source_western_date,source_event_type_chinese,source_event_name,source_event_description,citation,evtp_id,place_id,attestation
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [46]:
print(downstream.shape)
downstream = downstream.filter(
    (pl.col("latitude").is_not_null()).and_(pl.col("longitude").is_not_null())
)

(1556, 10)


In [47]:
ev_df2.head(1)

event_id,event_date_western,event_date_ch,event_description,event_notes,evc_id,event_type_ch,event_type_py,event_type_en,type_category_ch,type_category_en,source,source_page,source_ch_date,source_western_date,source_event_type_chinese,source_event_name,source_event_description,citation,evtp_id,place_id,attestation
str,f64,str,str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],str,str,str,str
"""ev_2015""",1588.0,"""明神宗万历十六年""",null,null,"[""evc_2""]","[""議""]","[""yi""]","[""Proposals and Discussion""]","[""水利""]","[""Management""]","[""HHNB""]",[null],"[""明神宗万历十六年""]","[""1588""]","[""条陈""]",[null],"[""六月己未勘科常居敬奏：“黄河故道开复甚难，宜罢役。”而訾家营支河之议起。（明神宗实录）""]","""HHNB, in 1588 (明神宗万历十六年) descr…","""evtp_8822""","""yrdb2772""","""ds_872"""


In [48]:
yrdb_events = []

for entry in downstream.iter_rows():
    yrdb_id = entry[0]
    tr_title = entry[1]
    ch_pinyin = entry[2]
    lat = entry[3]
    long = entry[4]
    class_en = entry[7]
    type_ch = entry[6]
    type_en = entry[8]
    events = []
    for event in ev_df2.filter(pl.col("place_id") == yrdb_id).iter_rows():
        event_id = event[0]
        en_date_start = event[1]
        ch_date = event[2]
        en_cat = event[10]
        en_type = event[8]
        en_title = event[7]
        ch_cat = event[9]
        ch_title = event[6]
        description = event[17]
        citation = event[18]
        source = event[11]
        src_page = event[12]
        src_ch_date = event[13]
        src_west_date = event[14]

        event_dict = {
            "event_id": event_id,
            "en_date_start": en_date_start,
            "ch_date": ch_date,
            "en_cat": en_cat,
            "en_type": en_type,
            "en_title": en_title,
            "ch_cat": ch_cat,
            "ch_title": ch_title,
            "description": description,
            "citation": citation,
            "source": source,
            "src_page": src_page,
            "src_ch_date": src_ch_date,
            "src_west_date": src_west_date,
        }
        events.append(event_dict)

    # Build place dictionary with nested events
    place_dict = {
        "yrdb_id": yrdb_id,
        "tr_title": tr_title,
        "ch_pinyin": ch_pinyin,
        "lat": lat,
        "long": long,
        "class_en": class_en,
        "type_ch": type_ch,
        "type_en": type_en,
        "events": events,
    }
    yrdb_events.append(place_dict)

# Write to JSON file
with open("yrdb_events.json", "w", encoding="utf-8") as f:
    json.dump(yrdb_events, f, ensure_ascii=False, indent=2)